# 11 -- LangGraph Agents
**Atlas context**: tool-using agent that dispatches techs, runs reports, triggers campaigns. Shows state machine, confirm-before-execute, streaming.

In [1]:
from typing import Literal, Optional, TypedDict
from langgraph.graph import StateGraph, END
import json, time, re
import langgraph; print('LangGraph installed OK')

LangGraph installed OK


## 1. Tool Registry

In [2]:
JOBS = [
    {'id':'J001','customer':'Smith HVAC',    'status':'pending',  'tech':None,'value':850},
    {'id':'J002','customer':'Jones Plumbing','status':'pending',  'tech':None,'value':1200},
    {'id':'J003','customer':'Brown Electric','status':'completed','tech':'T01','value':650},
]
TECHS = [
    {'id':'T01','name':'Carlos M.','available':False,'skill':'HVAC',   'jobs':3},
    {'id':'T02','name':'Priya S.', 'available':True, 'skill':'Plumbing','jobs':1},
    {'id':'T03','name':'James K.','available':True, 'skill':'HVAC',   'jobs':2},
]

def get_pending_jobs():     return [j for j in JOBS  if j['status']=='pending']
def get_available_techs():  return [t for t in TECHS if t['available']]
def get_schedule_summary():
    p = sum(1 for j in JOBS if j['status']=='pending')
    a = sum(1 for t in TECHS if t['available'])
    return {'pending_jobs':p,'available_techs':a,'utilization':f'{(3-a)/3*100:.0f}%'}

def assign_tech(job_id, tech_id):
    for j in JOBS:
        if j['id']==job_id:
            j['tech'],j['status'] = tech_id,'assigned'
            name = next(t['name'] for t in TECHS if t['id']==tech_id)
            return f'Assigned {name} to {job_id} ({j["customer"]})'
    return f'Job {job_id} not found'

TOOLS = {
    'get_pending_jobs':     (get_pending_jobs,     'read'),
    'get_available_techs':  (get_available_techs,  'read'),
    'get_schedule_summary': (get_schedule_summary, 'read'),
    'assign_tech':          (assign_tech,           'write'),
}
print('Read: ', [k for k,v in TOOLS.items() if v[1]=='read'])
print('Write:', [k for k,v in TOOLS.items() if v[1]=='write'])

Read:  ['get_pending_jobs', 'get_available_techs', 'get_schedule_summary']
Write: ['assign_tech']


## 2. Mock LLM (Function Calling Pattern)

In [3]:
def mock_llm(msg):
    # In production: Azure OpenAI with function_calling schema
    m = msg.lower()
    if 'pending' in m or 'unassigned' in m:
        return {'tool':'get_pending_jobs','args':{}}
    if 'available' in m:
        return {'tool':'get_available_techs','args':{}}
    if 'schedule' in m or 'summary' in m or 'overview' in m:
        return {'tool':'get_schedule_summary','args':{}}
    if 'assign' in m or 'dispatch' in m:
        jm = re.search(r'J\d{3}', msg.upper())
        tm = re.search(r'T\d{2}', msg.upper())
        return {'tool':'assign_tech',
                'args':{'job_id':  jm.group() if jm else 'J001',
                        'tech_id': tm.group() if tm else 'T02'}}
    return {'tool':None,'args':{}}

for q in ['Show pending jobs','Who is available?','Assign T02 to J001']:
    print(f'  {q!r} -> {mock_llm(q)}')

  'Show pending jobs' -> {'tool': 'get_pending_jobs', 'args': {}}
  'Who is available?' -> {'tool': 'get_available_techs', 'args': {}}
  'Assign T02 to J001' -> {'tool': 'assign_tech', 'args': {'job_id': 'J001', 'tech_id': 'T02'}}


## 3. Agent State + Nodes

In [4]:
class S(TypedDict):
    msg:     str
    intent:  Optional[dict]
    result:  Optional[object]
    pending: Optional[dict]
    confirm: Optional[bool]
    answer:  Optional[str]
    trace:   list

def parse(s):
    return {'intent':mock_llm(s['msg']),'trace':s['trace']+['parse']}

def read_tool(s):
    fn,_ = TOOLS[s['intent']['tool']]
    a = s['intent']['args']
    return {'result': fn(**a) if a else fn(),'trace':s['trace']+['read']}

def stage_write(s):
    t,a = s['intent']['tool'], s['intent']['args']
    print(f'  CONFIRM REQUIRED: {t}({a})')
    return {'pending':{'tool':t,'args':a},'trace':s['trace']+['stage']}

def exec_confirmed(s):
    if not s.get('confirm'):
        return {'answer':'Action cancelled.','trace':s['trace']+['cancel']}
    c = s['pending']
    fn,_ = TOOLS[c['tool']]
    return {'result':fn(**c['args']),'trace':s['trace']+['write']}

def fmt(s):
    r = s.get('result')
    if r is None:         ans = 'I help with scheduling and dispatch.'
    elif isinstance(r,list): ans = f'{len(r)} results: '+'; '.join(str(x) for x in r)
    elif isinstance(r,dict): ans = ' | '.join(f'{k}:{v}' for k,v in r.items())
    else:                 ans = str(r)
    return {'answer':ans,'trace':s['trace']+['format']}

def route(s) -> Literal['read','write','reply']:
    t = s['intent'].get('tool')
    if not t: return 'reply'
    return TOOLS[t][1] if TOOLS[t][1]=='write' else 'read'

print('Nodes OK')

Nodes OK


## 4. Compile Graph

In [5]:
g = StateGraph(S)
for name,fn in [('parse',parse),('read',read_tool),
                ('stage',stage_write),('exec',exec_confirmed),('fmt',fmt)]:
    g.add_node(name, fn)
g.set_entry_point('parse')
g.add_conditional_edges('parse', route, {'read':'read','write':'stage','reply':'fmt'})
g.add_edge('read','fmt'); g.add_edge('stage',END)
g.add_edge('exec','fmt'); g.add_edge('fmt',END)
agent = g.compile()
print('Compiled. Paths:')
print('  READ  -> read -> fmt -> END')
print('  WRITE -> stage -> END  [pause for confirm]')
print('  OTHER -> fmt -> END')

Compiled. Paths:
  READ  -> read -> fmt -> END
  WRITE -> stage -> END  [pause for confirm]
  OTHER -> fmt -> END


## 5. Read Queries

In [6]:
def run(msg, confirm=None):
    return agent.invoke(S(msg=msg,intent=None,result=None,
                          pending=None,confirm=confirm,answer=None,trace=[]))

for q in ['Show me pending jobs','What is the schedule overview?']:
    r = run(q)
    print(f'Q: {q}')
    print(f'A: {r["answer"]}')
    print(f'Trace: {" -> ".join(r["trace"])}\n')

Q: Show me pending jobs
A: 2 results: {'id': 'J001', 'customer': 'Smith HVAC', 'status': 'pending', 'tech': None, 'value': 850}; {'id': 'J002', 'customer': 'Jones Plumbing', 'status': 'pending', 'tech': None, 'value': 1200}
Trace: parse -> read -> format

Q: What is the schedule overview?
A: pending_jobs:2 | available_techs:2 | utilization:33%
Trace: parse -> read -> format



## 6. Write Op -- Confirm-Before-Execute

In [7]:
r = run('Assign T02 to J001')
print(f'Paused at: {r["pending"]}')

g2 = StateGraph(S)
g2.add_node('exec',exec_confirmed); g2.add_node('fmt',fmt)
g2.set_entry_point('exec')
g2.add_edge('exec','fmt'); g2.add_edge('fmt',END)
exec_g = g2.compile()

print('\n--- CONFIRM ---')
f = exec_g.invoke(S(**{**r,'confirm':True}))
print(f'Result : {f["answer"]}')

print('\n--- CANCEL ---')
f2 = exec_g.invoke(S(**{**r,'confirm':False}))
print(f'Result : {f2["answer"]}')

  CONFIRM REQUIRED: assign_tech({'job_id': 'J001', 'tech_id': 'T02'})
Paused at: {'tool': 'assign_tech', 'args': {'job_id': 'J001', 'tech_id': 'T02'}}

--- CONFIRM ---
Result : Assigned Priya S. to J001 (Smith HVAC)

--- CANCEL ---
Result : I help with scheduling and dispatch.


## 7. Streaming Simulation

In [8]:
def stream(msg):
    r = run(msg)
    print(f'User : {msg}')
    print('Atlas: ', end='', flush=True)
    for w in (r['answer'] or '').split():
        print(w+' ', end='', flush=True)
        time.sleep(0.04)
    print()

# Production: agent.astream() + Azure OpenAI stream=True
stream('Show me all available technicians')
print()
stream('What is the current schedule overview?')

User : Show me all available technicians
Atlas: 

2 

results: 

{'id': 

'T02', 

...

pending_jobs:1 

| 

available_techs:2 

| 

utilization:33% 

## 8. Design Tradeoffs

In [9]:
rows = [
    ('Rule-based routing',     'O(k) deterministic',   'Brittle on edge cases'),
    ('LLM-based routing',      'Handles ambiguity',    'Latency + cost per call'),
    ('Confirm-before-execute', 'Safe, auditable',      'Extra round-trip for writes'),
    ('Immediate execution',    'Fast UX',              'Catastrophic if write errors'),
    ('StateGraph',             'Explicit, cyclable',   'More setup vs simple chains'),
]
print(f'  {"Pattern":<28} {"Pro":<30} Con')
print('-'*75)
for r,p,c in rows:
    print(f'  {r:<28} {p:<30} {c}')
print()
print('COMPLEXITY')
print('  Intent:   O(k) rules | O(1) LLM call')
print('  Tool:     O(log n) with DB index')
print('  Graph:    O(V+E) per run -- tiny for agent graphs')
print('  Memory:   O(log n) ANN retrieval')

  Pattern                      Pro                            Con
---------------------------------------------------------------------------
  Rule-based routing           O(k) deterministic             Brittle on edge cases
  LLM-based routing            Handles ambiguity              Latency + cost per call
  Confirm-before-execute       Safe, auditable                Extra round-trip for writes
  Immediate execution          Fast UX                        Catastrophic if write errors
  StateGraph                   Explicit, cyclable             More setup vs simple chains

COMPLEXITY
  Intent:   O(k) rules | O(1) LLM call
  Tool:     O(log n) with DB index
  Graph:    O(V+E) per run -- tiny for agent graphs
  Memory:   O(log n) ANN retrieval
